In [34]:
from tdc.single_pred import Tox

data = Tox(name='hERG_Karim')
df_raw = data.get_data()
print(df_raw.shape)
df_raw.head()

Found local copy...
Loading...
Done!


(13445, 3)


,Drug_ID,Drug,Y
0,0,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,1
1,1,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,0
2,2,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,0
3,3,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,0
4,4,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,0


In [35]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13445 entries, 0 to 13444
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Drug_ID  13445 non-null  int64 
 1   Drug     13445 non-null  object
 2   Y        13445 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 315.2+ KB


In [36]:
df_raw['Y'].value_counts(normalize=True)

Y
0    0.500335
1    0.499665
Name: proportion, dtype: float64

In [37]:
df=df_raw.drop(columns=['Drug_ID']).copy()
df.head()

,Drug,Y
0,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,1
1,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,0
2,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,0
3,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,0
4,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,0


The raw dataset had 3 columns: Drug_ID, Drug (molecular structure as a SMILES string) and Y, which is a target label. Target indicates whether the molecule blocks the hERG potassium channel (Y=1,IC50 < 10µM) or not (Y=0, IC50 >= 10 µM)  Drug_ID column was removed, because it does not contain
 usefull information for prediction.

The dataset contains 13 445 molecules with no missing values and classes are well balanced.

In [38]:
# Scaffold split (70% train, 15% validation, 15% test)
split = data.get_split(method='scaffold', frac=[0.7, 0.15, 0.15], seed=42)

train_df = split['train']
val_df   = split['valid']
test_df  = split['test']


y_train = train_df['Y'].values
y_val   = val_df['Y'].values
y_test  = test_df['Y'].values

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)


100%|██████████| 13445/13445 [00:04<00:00, 2773.29it/s]

Training set: (9411, 3)
Validation set: (2016, 3)
Test set: (2018, 3)


FEATURE SELECTION

Each SMILES string is converted into numerical feature vector using RDKit descriptor featurizer, which calculates a set of descriptors, 210 odf them, per molecule

In [39]:
from deepchem.feat import RDKitDescriptors

featurizer = RDKitDescriptors()

Xtr = featurizer.featurize(train_df['Drug'].tolist())   # train
Xva = featurizer.featurize(val_df['Drug'].tolist())     # val
Xte = featurizer.featurize(test_df['Drug'].tolist())    # test

In [40]:
print(f"Number of generated molecular descriptors: {Xtr.shape[1]}")

Number of generated molecular descriptors: 210


We need to check the descriptors for invalid vales, infinity values are converted to NanN and any descriptor with invalid values is removed. After that we remove zero variance descriptors, since they carry no information. This reduces the number of features to 190

In [41]:
import numpy as np

Xtr = np.where(np.isinf(Xtr), np.nan, Xtr)
valid_cols = ~np.isnan(Xtr).any(axis=0)

Xtr = Xtr[:, valid_cols]
Xva = np.where(np.isinf(Xva), np.nan, Xva)[:, valid_cols]
Xte = np.where(np.isinf(Xte), np.nan, Xte)[:, valid_cols]

print(f"Number of molecular descriptors without invalid values: {Xtr.shape[1]}")

Number of molecular descriptors without invalid values: 198


In [42]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.0)
Xtr = selector.fit_transform(Xtr)
Xva = selector.transform(Xva)
Xte = selector.transform(Xte)

print(f"Number of features after removing zero-variance features: {Xtr.shape[1]}")

Number of features after removing zero-variance features: 190


The features are standardized with StandardScaler, which was fitted on the training set and applied to the validation and test sets

In [43]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xva_s = scaler.transform(Xva)
Xte_s = scaler.transform(Xte)

print("Scaled train:", Xtr_s.shape)
print("Scaled val:  ", Xva_s.shape)
print("Scaled test: ", Xte_s.shape)

Scaled train: (9411, 190)
Scaled val:   (2016, 190)
Scaled test:  (2018, 190)


For feature selection we use L1 regularized logistic regression, which penalizes large coefficients and sets least informative ones to 0, and by doing this it removes those features. Only selected descriptors, 123 of them, are kept and applied to all three sets.

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)
selector_l1 = SelectFromModel(l1)

selector_l1.fit(Xtr_s, y_train)       

Xtr_sel = selector_l1.transform(Xtr_s)  
Xva_sel = selector_l1.transform(Xva_s)
Xte_sel = selector_l1.transform(Xte_s)

print(f"Features selected by L1: {Xtr_sel.shape[1]} / {Xtr_s.shape[1]}")

Features selected by L1: 123 / 190


RANDOM FOREST

Hyperparameters are tuned with 5 fold cross validation on the training set, optimizing ROC AUC, and the best configuration is evaluated on the validation set.

In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_grid = {
    'n_estimators':      [100, 150, 200, 250],
    'max_depth':         [None, 10, 20, 30],
    'max_features':      ['sqrt', 'log2', 0.5],
    'min_samples_leaf':  [1, 2, 5],
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = GridSearchCV(rf, rf_grid, scoring='roc_auc', cv=5, n_jobs=-1, verbose=1)
rf_search.fit(Xtr_sel, y_train)

print("RF best params:", rf_search.best_params_)
print(f"RF best CV ROC-AUC: {rf_search.best_score_:.3f}")
best_rf = rf_search.best_estimator_

Fitting 5 folds for each of 144 candidates, totalling 720 fits


/opt/anaconda3/envs/cheminfo/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/envs/cheminfo/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/envs/cheminfo/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or

RF best params: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 250}
RF best CV ROC-AUC: 0.841


In [47]:
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             matthews_corrcoef, classification_report, confusion_matrix)

y_val_pred  = best_rf.predict(Xva_sel)
y_val_proba = best_rf.predict_proba(Xva_sel)[:, 1]

print("Tuned Random Forest — validation performance")
print(f"Accuracy : {accuracy_score(y_val, y_val_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, y_val_proba):.3f}")
print(f"F1       : {f1_score(y_val, y_val_pred):.3f}")
print(f"MCC      : {matthews_corrcoef(y_val, y_val_pred):.3f}")

Tuned Random Forest — validation performance
Accuracy : 0.759
ROC-AUC  : 0.841
F1       : 0.753
MCC      : 0.522
